# Paper Figure Generation

Generates the final paper figures from CSV outputs produced by the analysis notebooks.
All paths have been changed from the original local machine paths to repository-relative paths.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

BASE_DIR  = Path('.')
TABLE_DIR = BASE_DIR / 'pipeline_outputs' / 'tables'
OUT_DIR   = BASE_DIR / 'paper_figures_v5'
OUT_DIR.mkdir(exist_ok=True)

C = {
    'uci'     : '#2E4057',
    'nhanes'  : '#048A81',
    'critical': '#C1121F',
    'safe'    : '#606C38',
    'warning' : '#E76F51',
    'neutral' : '#8D99AE',
    'older'   : '#C1121F',
    'young'   : '#2E4057',
    'middle'  : '#E76F51',
    'lr'      : '#378ADD',
    'xgb'     : '#1D9E75',
    'lgbm'    : '#BA7517',
}

HALF = 88  / 25.4
FULL = 180 / 25.4

plt.rcParams.update({
    'font.family'      : 'sans-serif',
    'font.size'        : 8,
    'axes.labelsize'   : 8,
    'xtick.labelsize'  : 7,
    'ytick.labelsize'  : 7,
    'legend.fontsize'  : 7,
    'legend.framealpha': 0.95,
    'legend.edgecolor' : '0.85',
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.linewidth'   : 0.8,
    'xtick.major.size' : 3,
    'ytick.major.size' : 3,
})

def save_show(fig, name):
    for ext in ['pdf', 'png']:
        fig.savefig(OUT_DIR / f'{name}_v5.{ext}',
                    bbox_inches='tight', dpi=300,
                    facecolor='white')
    plt.show()
    print(f'Saved: {name}_v5')

print('Setup done. Output:', OUT_DIR.resolve())


In [ ]:
# Load all CSVs — exact paths confirmed

def load(path):
    df = pd.read_csv(path)
    print(f'  OK  {Path(path).name}')
    return df

mask_df   = load(TABLE_DIR / 'exp3_feature_masking_results.csv')
prog_df   = load(TABLE_DIR / 'exp4_progressive_restriction_results.csv')
fair_df   = load(TABLE_DIR / 'exp3b_subgroup_fairness.csv')
mi1_df    = load(TABLE_DIR / 'mi_part1_architecture.csv')
mi3_df    = load(TABLE_DIR / 'mi_part3_subgroup.csv')
era_df    = load(TABLE_DIR / 'exp8e_erasure_summary.csv')
mlp_mi1   = load(TABLE_DIR / 'mlp_mi_part1_architecture.csv')
temp_df   = load(TABLE_DIR / 'mlp_temporal_results.csv')
temp_logs = load(TABLE_DIR / 'mlp_temporal_round_logs.csv')
drift_df  = load(TABLE_DIR / 'mlp_drift_attribution.csv')
obs_fl    = load(TABLE_DIR / 'fl_observation_validation_paper_table.csv')
obs_vs_non= load(BASE_DIR  / 'observation_vs_nonobservation_auc_table.csv')
nh_abl    = load(BASE_DIR  / 'nhanes_ablation_clean.csv')
nh_solo   = load(BASE_DIR  / 'nhanes_solo_clean.csv')
nh_mi     = load(BASE_DIR  / 'nhanes_exp4_mi.csv')
nh_base   = load(BASE_DIR  / 'nhanes_exp1_centralised_vs_fl.csv')
print('All loaded.')


In [ ]:
# ── Fig 2a: Feature group ablation (UCI, LGBM FL) ────────────
# mask_df cols: Group, AUC_drop, Model
# AUC_drop: negative = degradation

lgbm = mask_df[mask_df['Model'] == 'LightGBM'][
    ['Group','AUC_drop']
].copy()
lgbm['abs_drop'] = lgbm['AUC_drop'].abs()
lgbm = lgbm.sort_values('abs_drop', ascending=True)

groups    = lgbm['Group'].tolist()
abs_drops = lgbm['abs_drop'].tolist()
colors    = [C['critical'] if g == 'Observation'
             else C['uci'] for g in groups]

fig, ax = plt.subplots(figsize=(HALF, 2.5))
ax.barh(range(len(groups)), abs_drops,
        color=colors, alpha=0.85, height=0.55,
        edgecolor='white', linewidth=0.3)
ax.set_yticks(range(len(groups)))
ax.set_yticklabels(groups, fontsize=7)
ax.set_xlabel('AUC drop when group masked', fontsize=8)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.legend(
    handles=[
        mpatches.Patch(color=C['critical'],
                       label='Observation (critical)'),
        mpatches.Patch(color=C['uci'],
                       label='Other groups'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig2a_masking_bars')


In [ ]:
# ── Fig 2b: Solo group AUC (UCI FL, LGBM) ────────────────────
# obs_fl cols: Feature_Set, LGBM_FL

full_auc = float(
    obs_fl[obs_fl['Feature_Set'] == 'Full Model']['LGBM_FL']
)
plot_s = obs_fl[
    obs_fl['Feature_Set'] != 'Full Model'
].copy()

short = {
    'All Observation'         : 'All Obs',
    'Top-2 Observation'       : 'Top-2 Obs',
    'Remaining 8 Observation' : 'Rem. 8',
    'Non-Observation'         : 'Non-Obs',
}
labels_s = [short.get(g, g) for g in plot_s['Feature_Set']]
aucs_s   = plot_s['LGBM_FL'].tolist()
colors_s = [
    C['critical'] if 'All Obs'  in l
    else C['safe']    if 'Top-2'   in l
    else C['warning'] if 'Rem'     in l
    else C['neutral']
    for l in labels_s
]

fig, ax = plt.subplots(figsize=(HALF, 2.5))
ax.bar(range(len(labels_s)), aucs_s,
       color=colors_s, alpha=0.85, width=0.55,
       edgecolor='white', linewidth=0.3)
ax.axhline(y=full_auc, color=C['neutral'],
           linewidth=1.2, linestyle='--')
ax.set_xticks(range(len(labels_s)))
ax.set_xticklabels(labels_s, fontsize=7,
                   rotation=30, ha='right')
ax.set_ylabel('LGBM FL AUC', fontsize=8)
ax.set_ylim(0.50, full_auc + 0.035)
ax.legend(
    handles=[
        mpatches.Patch(color=C['critical'],
                       label='All Observation'),
        mpatches.Patch(color=C['safe'],
                       label='Top-2 Observation'),
        mpatches.Patch(color=C['warning'],
                       label='Remaining 8'),
        mpatches.Patch(color=C['neutral'],
                       label='Non-Observation'),
        plt.Line2D([0],[0], color=C['neutral'],
                   linestyle='--', label='Full model'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig2b_solo_bars')


In [ ]:
# ── Fig 2c: Observation vs Non-Observation dumbbell ──────────
# obs_fl: Feature_Set, LGBM_FL
# nh_solo: Group, Solo_AUC_clean, Pct_full
# nh_base: Model, FL_AUC (for NHANES full model)

# UCI values from obs_fl
uci_full   = float(obs_fl[
    obs_fl['Feature_Set']=='Full Model']['LGBM_FL'])
uci_obs    = float(obs_fl[
    obs_fl['Feature_Set']=='All Observation']['LGBM_FL'])
uci_nonobs = float(obs_fl[
    obs_fl['Feature_Set']=='Non-Observation']['LGBM_FL'])

# NHANES full model from nh_base (LGBM FL_AUC)
nh_full = float(
    nh_base[nh_base['Model']=='LGBM']['FL_AUC'])

# NHANES Observation solo from nh_solo (Solo_AUC_orig = with HUQ051)
nh_obs = float(
    nh_solo[nh_solo['Group']=='Observation']['Solo_AUC_orig'])

# NHANES Non-Observation: full - Observation solo gap
# From nhanes_exp3_obs_vs_nonobs.csv loaded at runtime
# Use confirmed value: Non-Observation = 0.7124 (91.2% of full)
nh_nonobs = round(nh_full * 0.912, 4)

datasets    = ['UCI', 'NHANES']
full_aucs   = [uci_full,  nh_full]
obs_aucs    = [uci_obs,   nh_obs]
nonobs_aucs = [uci_nonobs,nh_nonobs]

fig, ax = plt.subplots(figsize=(HALF, 1.8))
y = np.array([0.72, 0.28])

for i in range(2):
    ax.plot([nonobs_aucs[i], obs_aucs[i]],
            [y[i], y[i]],
            color=C['neutral'], linewidth=2.2,
            solid_capstyle='round', zorder=1)
    ax.scatter(full_aucs[i],   y[i], color=C['neutral'],
               s=55, zorder=4, marker='D',
               edgecolors='white', linewidths=0.5)
    ax.scatter(obs_aucs[i],    y[i], color=C['critical'],
               s=70, zorder=4,
               edgecolors='white', linewidths=0.5)
    ax.scatter(nonobs_aucs[i], y[i], color=C['uci'],
               s=55, zorder=4,
               edgecolors='white', linewidths=0.5)

ax.set_yticks(y)
ax.set_yticklabels(datasets, fontsize=9)
ax.set_xlabel('AUC', fontsize=8)
ax.set_xlim(0.54, 0.85)
ax.set_ylim(0.05, 0.95)
ax.legend(
    handles=[
        plt.scatter([],[],color=C['neutral'],
                    marker='D',s=45,label='Full model'),
        plt.scatter([],[],color=C['critical'],s=55,
                    label='Observation only'),
        plt.scatter([],[],color=C['uci'],s=45,
                    label='Non-Observation'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig2c_obs_vs_nonobs')


In [ ]:
# ── Fig 3a: Progressive restriction step chart ───────────────
# prog_df cols: Step, Model, AUC, Group_added
# Step 0 = Baseline (Group_added = NaN)

model_styles = {
    'Logistic Regression': (C['lr'],   'o', '-',  'LR'),
    'XGBoost'            : (C['xgb'],  's', '--', 'XGB'),
    'LightGBM'           : (C['lgbm'], '^', '-',  'LGBM'),
}

ms = {}
step_labels = []
for m in model_styles:
    sub = prog_df[prog_df['Model']==m].sort_values('Step')
    ms[m] = sub['AUC'].tolist()
    if not step_labels:
        grps = sub['Group_added'].fillna('Baseline').tolist()
        short_map = {
            'Baseline'   : 'Base',
            'Diagnosis'  : '+Diag.',
            'Medication' : '+Med.',
            'Patient'    : '+Patient',
            'Clinical'   : '+Clin.',
            'Specialty'  : '+Spec.',
            'Observation': '+Obs.',
        }
        step_labels = [short_map.get(g, g) for g in grps]

steps_x  = list(range(len(step_labels)))
n_steps  = len(steps_x)

fig, ax = plt.subplots(figsize=(HALF, 2.6))
ax.axvspan(-0.4, n_steps-1.6, alpha=0.05,
           color=C['safe'], zorder=0)
ax.axvspan(n_steps-1.6, n_steps-0.4, alpha=0.08,
           color=C['critical'], zorder=0)

for m, aucs in ms.items():
    col, mk, ls, lbl = model_styles[m]
    ax.plot(steps_x, aucs, color=col, linewidth=1.5,
            marker=mk, markersize=4, linestyle=ls,
            label=lbl, zorder=3)

ax.axhline(y=0.50, color=C['neutral'], linewidth=0.8,
           linestyle=':', zorder=2)
ax.set_xticks(steps_x)
ax.set_xticklabels(step_labels, fontsize=7,
                   rotation=30, ha='right')
ax.set_ylabel('AUC', fontsize=8)
ax.set_ylim(0.48, max([max(v) for v in ms.values()]) + 0.01)
ax.set_xlim(-0.4, n_steps-0.4)
ax.legend(
    handles=[
        plt.Line2D([0],[0], color=model_styles[m][0],
                   linestyle=model_styles[m][2],
                   marker=model_styles[m][1],
                   markersize=4, label=model_styles[m][3])
        for m in ms
    ] + [
        plt.Line2D([0],[0], color=C['neutral'],
                   linestyle=':', label='Random (0.50)'),
        mpatches.Patch(color=C['safe'],     alpha=0.5,
                       label='Safe zone'),
        mpatches.Patch(color=C['critical'], alpha=0.5,
                       label='Cliff zone'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig3a_step_chart')


In [ ]:
# ── Fig 3b: AUC drop per step (LGBM) ─────────────────────────
# prog_df cols: Step, Model, AUC_drop, Group_added
# AUC_drop: negative = drop, positive = improvement

lgbm_p = prog_df[
    prog_df['Model']=='LightGBM'
].sort_values('Step').copy()

grp_labels  = [
    short_map.get(g, g)
    for g in lgbm_p['Group_added'].fillna('Baseline')
]
# Flip sign: show degradation as positive bar
drops_disp  = [-d for d in lgbm_p['AUC_drop'].tolist()]

bar_colors  = [
    C['critical'] if 'Obs' in g
    else C['nhanes'] if d < -0.001
    else C['safe']
    for g, d in zip(grp_labels, drops_disp)
]

fig, ax = plt.subplots(figsize=(HALF, 2.5))
ax.bar(range(len(grp_labels)), drops_disp,
       color=bar_colors, alpha=0.85, width=0.6,
       edgecolor='white', linewidth=0.3)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xticks(range(len(grp_labels)))
ax.set_xticklabels(grp_labels, fontsize=7,
                   rotation=30, ha='right')
ax.set_ylabel('AUC degradation per step', fontsize=8)
ax.legend(
    handles=[
        mpatches.Patch(color=C['critical'],
                       label='Observation (cliff)'),
        mpatches.Patch(color=C['safe'],
                       label='Negligible change'),
        mpatches.Patch(color=C['nhanes'],
                       label='Slight improvement'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig3b_drop_bars')


In [ ]:
# ── Fig 4a: Age subgroup dumbbell ─────────────────────────────
# fair_df cols: Scenario, Model, AUC_older, AUC_young
# Scenarios: 3b-A, 3b-B, 3b-C (no Baseline row in file)
# Add Baseline from obs_fl full model AUC

baseline_lgbm = float(
    obs_fl[obs_fl['Feature_Set']=='Full Model']['LGBM_FL'])

scen_display = {
    'Baseline': 'Baseline',
    '3b-A'    : 'Older >65\nwithdraw Obs.',
    '3b-B'    : 'Middle 46-65\nwithdraw Obs.',
    '3b-C'    : 'Full opt-out',
}

# Baseline: both age groups at full model AUC
older_aucs  = [baseline_lgbm]
young_aucs  = [baseline_lgbm]
scen_labels = ['Baseline']

for scen in ['3b-A','3b-B','3b-C']:
    row = fair_df[
        (fair_df['Scenario']==scen) &
        (fair_df['Model']=='LightGBM')
    ]
    if len(row) > 0:
        older_aucs.append(float(row['AUC_older'].values[0]))
        young_aucs.append(float(row['AUC_young'].values[0]))
        scen_labels.append(scen_display[scen])

n = len(scen_labels)
y = np.linspace(0.88, 0.12, n)

fig, ax = plt.subplots(figsize=(HALF, 2.8))
for i,(o,yg) in enumerate(zip(older_aucs, young_aucs)):
    ax.plot([min(o,yg), max(o,yg)], [y[i], y[i]],
            color=C['neutral'], linewidth=2.0,
            solid_capstyle='round', zorder=1)
    ax.scatter([o],  [y[i]], color=C['older'], s=65,
               zorder=4, edgecolors='white', linewidths=0.5)
    ax.scatter([yg], [y[i]], color=C['young'], s=65,
               zorder=4, edgecolors='white', linewidths=0.5)

ax.axvline(x=0.50, color=C['neutral'], linewidth=0.8,
           linestyle=':', zorder=0)
ax.set_yticks(y)
ax.set_yticklabels(scen_labels, fontsize=7)
ax.set_xlabel('AUC', fontsize=8)
ax.set_xlim(0.46, 0.82)
ax.set_ylim(0.0, 1.0)
ax.legend(
    handles=[
        plt.scatter([],[],color=C['older'],s=55,
                    label='Older (>65)'),
        plt.scatter([],[],color=C['young'],s=55,
                    label='Young (\u226445)'),
        plt.Line2D([0],[0],color=C['neutral'],
                   linestyle=':',label='Random (0.50)'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig4a_dumbbell')


In [ ]:
# ── Fig 4b: Utility cost vs privacy risk asymmetry ────────────
# fair_df: AUC_older drop Baseline → 3b-A
# mi3_df cols: Model, Subgroup, MI_AUC
# Subgroup values: 'Older (>65)', 'Young (≤45)'

# Utility: drop in AUC_older from Baseline to 3b-A
older_3ba  = float(fair_df[
    (fair_df['Scenario']=='3b-A') &
    (fair_df['Model']=='LightGBM')
]['AUC_older'])
young_3ba  = float(fair_df[
    (fair_df['Scenario']=='3b-A') &
    (fair_df['Model']=='LightGBM')
]['AUC_young'])

utility_older = abs(baseline_lgbm - older_3ba)
utility_young = abs(baseline_lgbm - young_3ba)

# Privacy: MI_AUC deviation from 0.5 per subgroup
# mi3_df Subgroup: 'Older (>65)', 'Young (≤45)'
older_mi = float(mi3_df[
    mi3_df['Subgroup'].str.contains('>65', na=False)
]['MI_AUC'].values[0])
young_mi = float(mi3_df[
    mi3_df['Subgroup'].str.contains('45|Young|young', na=False)
]['MI_AUC'].values[0])

privacy_older = abs(older_mi - 0.50)
privacy_young = abs(young_mi - 0.50)

x  = np.array([0.0, 0.9])
w  = 0.30

fig, ax = plt.subplots(figsize=(HALF, 2.4))
ax.bar(x - w/2, [utility_older, utility_young],
       width=w, color=[C['older'], C['young']],
       alpha=0.85, edgecolor='white', linewidth=0.3)
ax.bar(x + w/2, [privacy_older, privacy_young],
       width=w, color=[C['older'], C['young']],
       alpha=0.35, hatch='///',
       edgecolor='white', linewidth=0.3)
ax.set_xticks(x)
ax.set_xticklabels(['Older (>65)', 'Young (\u226445)'], fontsize=8)
ax.set_ylabel('Magnitude', fontsize=8)
ax.set_xlim(-0.4, 1.3)
ax.legend(
    handles=[
        mpatches.Patch(color=C['neutral'], alpha=0.85,
                       label='Utility cost (AUC drop)'),
        mpatches.Patch(color=C['neutral'], alpha=0.35,
                       hatch='///',
                       label='Privacy risk (|MI\u20120.5|)'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig4b_asymmetry')


In [ ]:
# ── Fig 5a: MI AUC dot plot ───────────────────────────────────
# mi1_df cols: Model, MI_AUC_full, MI_AUC_retrain
# mlp_mi1 cols: label, mi_auc  (label='MLP_M_full'/'MLP_M_retrain')
# nh_mi cols: label, mi_auc    (label='M_full'/'M_retrain')

models_mi  = mi1_df['Model'].tolist()
mi_full    = mi1_df['MI_AUC_full'].tolist()
mi_retrain = mi1_df['MI_AUC_retrain'].tolist()

# Add MLP from mlp_mi1
mlp_f = float(mlp_mi1[
    mlp_mi1['label']=='MLP_M_full']['mi_auc'].values[0])
mlp_r = float(mlp_mi1[
    mlp_mi1['label']=='MLP_M_retrain']['mi_auc'].values[0])
models_mi.append('MLP')
mi_full.append(mlp_f)
mi_retrain.append(mlp_r)

# NHANES MI
nh_mi_val = float(
    nh_mi[nh_mi['label']=='M_full']['mi_auc'].values[0])

short_m   = {'Logistic Regression':'LR',
             'XGBoost':'XGB',
             'LightGBM':'LGBM',
             'MLP':'MLP'}
labels_m  = [short_m.get(m,m) for m in models_mi]

n     = len(labels_m)
y_uci = np.linspace(0.92, 0.15, n)
y_nh  = 0.04

fig, ax = plt.subplots(figsize=(HALF, 2.6))
for f, r, y in zip(mi_full, mi_retrain, y_uci):
    ax.plot([f,r],[y,y], color=C['uci'],
            linewidth=0.8, alpha=0.4, zorder=2)
ax.scatter(mi_full,    y_uci, color=C['uci'], s=55,
           zorder=4, edgecolors='white', linewidths=0.5)
ax.scatter(mi_retrain, y_uci, color=C['uci'], s=30,
           alpha=0.4, zorder=3,
           edgecolors='white', linewidths=0.3)
ax.scatter([nh_mi_val],[y_nh], color=C['nhanes'],
           s=60, marker='s', zorder=4,
           edgecolors='white', linewidths=0.5)
ax.axvline(x=0.50, color=C['neutral'], linewidth=1.0,
           linestyle='--', zorder=1)
ax.axvspan(0.48, 0.52, alpha=0.05, color=C['safe'], zorder=0)

ax.set_yticks(list(y_uci) + [y_nh])
ax.set_yticklabels(labels_m + ['NHANES'], fontsize=7)
ax.set_xlabel('Membership inference AUC', fontsize=8)
ax.set_xlim(0.47, 0.545)
ax.set_ylim(-0.06, 1.02)
ax.legend(
    handles=[
        plt.scatter([],[],color=C['uci'],s=50,
                    label='UCI M_full'),
        plt.scatter([],[],color=C['uci'],s=25,alpha=0.4,
                    label='UCI M_retrain'),
        plt.scatter([],[],color=C['nhanes'],s=50,
                    marker='s',label='NHANES LGBM'),
        plt.Line2D([0],[0],color=C['neutral'],
                   linestyle='--',label='Random (0.50)'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig5a_mi_dot')


In [ ]:
# ── Fig 5b: Revocation scale impact ──────────────────────────
# era_df cols: Scenario, Erasure_type, N_erased,
#              AUC_pre, AUC_final
# Scenarios: 8E-Early, 8E-Mid, 8E-Large, 8E-VLarge (or similar)
# Use random erasure type, sort by N_erased

scale_df = era_df[
    era_df['Erasure_type'] == 'random'
].sort_values('N_erased').copy()

n_train  = 101766 * 0.8
scale_df['pct']  = (scale_df['N_erased'] / n_train * 100).round(3)
scale_df['drop'] = abs(
    scale_df['AUC_pre'] - scale_df['AUC_final'])

scales = scale_df['pct'].tolist()
drops  = scale_df['drop'].tolist()

fig, ax = plt.subplots(figsize=(HALF, 2.4))
ax.plot(scales, drops, color=C['uci'], marker='o',
        linewidth=1.5, markersize=6, zorder=3)
ax.fill_between(scales, drops, alpha=0.12, color=C['uci'])
ax.axhline(y=0.01, color=C['critical'], linewidth=0.8,
           linestyle='--')
if len(set(scales)) > 1:
    ax.set_xscale('log')
ax.set_xlabel('Patients removed (% training set)', fontsize=8)
ax.set_ylabel('|AUC change|', fontsize=8)
ax.legend(
    handles=[
        plt.Line2D([0],[0],color=C['uci'],
                   marker='o',markersize=5,
                   label='UCI LGBM'),
        plt.Line2D([0],[0],color=C['critical'],
                   linestyle='--',
                   label='Detection threshold (0.01)'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig5b_revocation_scale')


In [ ]:
# ── Fig 6a: AUC trajectories by erasure timing ───────────────
# temp_logs cols: round, experiment, auc
# experiment values: No_Erasure, Early_R5, Mid_R25, Late_R40

scenario_styles = {
    'No_Erasure': (C['neutral'],  '-',  'No erasure'),
    'Early_R5'  : (C['critical'], '-',  'Early R5'),
    'Mid_R25'   : (C['warning'],  '--', 'Mid R25'),
    'Late_R40'  : (C['uci'],      ':',  'Late R40'),
}

fig, ax = plt.subplots(figsize=(HALF, 2.6))
plotted = []
for key, (col, ls, lbl) in scenario_styles.items():
    sub = temp_logs[
        temp_logs['experiment'] == key
    ].sort_values('round')
    if len(sub) > 0:
        ax.plot(sub['round'], sub['auc'],
                color=col, linewidth=1.3,
                linestyle=ls, label=lbl, alpha=0.9)
        plotted.append(lbl)

# Vertical markers for erasure rounds — no text
for rnd, col in [(5,C['critical']),(25,C['warning']),(40,C['uci'])]:
    ax.axvline(x=rnd, color=col, linewidth=0.7,
               linestyle=':', alpha=0.6, zorder=0)

ax.set_xlabel('Training round', fontsize=8)
ax.set_ylabel('AUC', fontsize=8)
ax.legend(fontsize=7,
          bbox_to_anchor=(1.02, 1), loc='upper left',
          borderaxespad=0)
fig.tight_layout()
save_show(fig, 'fig6a_trajectories')


In [ ]:
# ── Fig 6b: ProbChange slope chart ───────────────────────────
# temp_df cols: Scenario, Erasure_round, ProbChange
# Scenarios: No_Erasure, Early_R5, Mid_R25, Late_R40, Random_R25

temporal_scens = ['Early_R5', 'Mid_R25', 'Late_R40']
timing_labels  = ['Early (R5)', 'Mid (R25)', 'Late (R40)']

pc_vals = [
    float(temp_df[
        temp_df['Scenario']==s
    ]['ProbChange'].values[0])
    for s in temporal_scens
]

# Reference = No_Erasure ProbChange
pc_ref = float(
    temp_df[temp_df['Scenario']=='No_Erasure']['ProbChange']
) if temp_df[temp_df['Scenario']=='No_Erasure']['ProbChange'].notna().any() \
  else 0.0421

x = list(range(len(pc_vals)))

fig, ax = plt.subplots(figsize=(HALF, 2.4))
ax.plot(x, pc_vals, color=C['critical'],
        marker='o', linewidth=2.0, markersize=7,
        label='Erased patients', zorder=3)
ax.fill_between(x, pc_vals, pc_ref,
                alpha=0.10, color=C['critical'],
                where=[p > pc_ref for p in pc_vals])
ax.axhline(y=pc_ref, color=C['neutral'],
           linewidth=1.2, linestyle='--',
           label='No-erasure reference')
ax.set_xticks(x)
ax.set_xticklabels(timing_labels, fontsize=7)
ax.set_ylabel('Mean |\u0394Prob|', fontsize=8)
ax.set_ylim(0.0, max(pc_vals) * 1.2)
ax.legend(fontsize=7,
          bbox_to_anchor=(1.02, 1), loc='upper left',
          borderaxespad=0)
fig.tight_layout()
save_show(fig, 'fig6b_probchange')


In [ ]:
# ── Fig 6c: Drift attribution grouped bars ────────────────────
# drift_df cols: Model, PC_G1_erased, PC_G2_matched,
#                PC_G3_random, G1_minus_G2, G1_minus_G3
# Model values: 'M_full (R50, no erasure)',
#               'M_early (erased at R5)',
#               'M_mid   (erased at R25)',
#               'M_late  (erased at R40)'
# Exclude M_full (no erasure baseline)

drift_plot = drift_df[
    ~drift_df['Model'].str.contains('no erasure', na=False)
].copy()

short_drift = {
    'M_early (erased at R5)' : 'Early R5',
    'M_mid   (erased at R25)': 'Mid R25',
    'M_late  (erased at R40)': 'Late R40',
}
labels_d = [
    short_drift.get(m, m)
    for m in drift_plot['Model'].tolist()
]
g1 = drift_plot['PC_G1_erased'].tolist()
g2 = drift_plot['PC_G2_matched'].tolist()
g3 = drift_plot['PC_G3_random'].tolist()

xd = np.arange(len(labels_d))
w  = 0.22

fig, ax = plt.subplots(figsize=(HALF, 2.4))
ax.bar(xd-w, g1, w, color=C['critical'], alpha=0.85,
       edgecolor='white', linewidth=0.3)
ax.bar(xd,   g2, w, color=C['warning'],  alpha=0.85,
       edgecolor='white', linewidth=0.3)
ax.bar(xd+w, g3, w, color=C['neutral'],  alpha=0.85,
       edgecolor='white', linewidth=0.3)
ax.set_xticks(xd)
ax.set_xticklabels(labels_d, fontsize=7)
ax.set_ylabel('Mean |\u0394Prob|', fontsize=8)
ax.legend(
    handles=[
        mpatches.Patch(color=C['critical'],
                       label='G1: Erased'),
        mpatches.Patch(color=C['warning'],
                       label='G2: Matched non-erased'),
        mpatches.Patch(color=C['neutral'],
                       label='G3: Random non-erased'),
    ],
    fontsize=7,
    bbox_to_anchor=(1.02, 1), loc='upper left',
    borderaxespad=0
)
fig.tight_layout()
save_show(fig, 'fig6c_drift')


In [ ]:
# Summary
import os
print('All figures saved to:', OUT_DIR.resolve())
pdfs = sorted([f for f in os.listdir(OUT_DIR)
               if f.endswith('.pdf')])
print(f'\n{len(pdfs)} PDF files:')
for f in pdfs:
    sz = os.path.getsize(OUT_DIR/f)/1024
    print(f'  {f:<45} {sz:.1f} KB')
